# Lab 7.12 &mdash; Capstone: The Email-Drafting Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Chain the pipeline: extract, route, gather (via the 7.11 agent's tool), draft, validate
- Never auto-send -- every result is a needs_approval draft
- Run it over a SUITE of client emails and score per-row (a partial score is honest)

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, and the pipeline logic &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; Now build it — Module 7 labs](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-07-12"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Capstone: the **email-drafting agent** (the client's Lab 4.1), end to end. It **extracts** the
query's fields, **routes** it to a team, **gathers** the order via the **gather-only agent you
assembled in Lab 7.11** (`send_email` is still withheld), **drafts** a grounded reply, **validates**
it, and returns a **`needs_approval`** draft &mdash; it **never auto-sends**. You run it over a
**suite of four incoming emails** &mdash; including an **unknown order** and an angry complaint the
keyword extractor mis-reads &mdash; and score it **per row**. A real eval yields a **partial** score;
that's honest, and it shows you exactly where the agent needs work. The helpers below are the ones you
built through the module; you assemble them into `process` and score with `evaluate`.

In [ ]:
from langchain_core.prompts import PromptTemplate

# The email agent's context sources: a small orders DB and a set of reply templates.
ORDERS = {
    "4471": {"id": "4471", "name": "Priya", "status": "shipped",    "eta": "Friday",    "carrier": "BlueDart"},
    "5090": {"id": "5090", "name": "Sam",   "status": "processing", "eta": "next week", "carrier": "-"},
}
TEMPLATES = {
    "delivery_delay": "Hi {name}, your order {id} has {status} and is due {eta}. Thanks for your patience.",
    "refund":         "Hi {name}, we've started the refund for order {id}. It will reflect in 5-7 days.",
}

from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

# --- The pipeline pieces you built this module (provided so you can assemble the whole agent) ---
def extract(email):
    text = email.lower()
    digits = "".join(ch for ch in email if ch.isdigit())
    order_id = digits if digits else None
    if "refund" in text: intent = "refund"
    elif any(w in text for w in ("deliver", "arrive", "late", "where is")): intent = "delivery"
    elif "cancel" in text: intent = "cancel"
    else: intent = "other"
    sentiment = "negative" if any(w in text for w in ("frustrated", "angry", "late", "still")) else "neutral"
    return {"order_id": order_id, "intent": intent, "sentiment": sentiment}

TEAMS = {"refund": "billing", "delivery": "logistics", "cancel": "billing", "other": "general"}
def route(rec):
    return {"team": TEAMS.get(rec["intent"], "general"),
            "escalate": rec["sentiment"] == "negative" or rec["intent"] not in TEAMS}

TEMPLATE_FOR = {"delivery": "delivery_delay", "refund": "refund"}
def draft(order, intent):
    kind = TEMPLATE_FOR.get(intent, "delivery_delay")
    return PromptTemplate.from_template(TEMPLATES[kind]).format(
        name=order["name"], id=order["id"], status=order["status"], eta=order["eta"])
def validate(reply, order):
    return order["eta"] in reply

# --- The gather-only agent you assembled in Lab 7.11 (send_email WITHHELD) -- reused here ---
@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order's status, ETA and carrier by id."""
    return ORDERS.get(order_id, {"status": "unknown"})
@tool
def get_template(kind: str) -> str:
    """Fetch a reply template by kind."""
    return TEMPLATES.get(kind, "")
@tool
def send_email(to: str, body: str) -> str:
    """Send an email. (Defined to show the capability -- but DELIBERATELY WITHHELD from the agent.)"""
    return "SENT"
def gather_tools():
    return [lookup_order, get_template]                 # gather-only -- send_email is NOT bound
def make_email_agent():
    return create_agent(ChatOllama(model="llama3.2:1b"), gather_tools())
print("helpers ready: extract, route, draft, validate + the gather-only agent (make_email_agent)")

## Your Turn
Assemble `process` (chain the pipeline; gather via the agent's tool; never send) and `evaluate`
(score the suite per row).

In [ ]:
def process(email):
    rec    = extract(email)
    routed = route(rec)
    # gather via the SAME tool the Lab 7.11 agent is built from (reuse, not re-implement)
    found  = lookup_order.invoke(rec["order_id"]) if rec["order_id"] else {}
    order  = found if found.get("id") else {"id": rec["order_id"], "name": "there", "status": "unknown", "eta": "soon"}
    reply  = ___   # TODO: draft a grounded reply for this order & intent
    ok     = validate(reply, order)
    # never auto-send: a valid draft awaits approval; an invalid one needs a fix
    status = ___   # TODO: "needs_approval" if ok else "needs_fix"
    return {"team": routed["team"], "escalate": routed["escalate"],
            "draft": reply, "status": status}

# Each row carries the human-labelled expected team + escalation for scoring.
SUITE = [
    {"email": "Where is my order 4471? It's late.",             "team": "logistics", "escalate": True},
    {"email": "Please refund order 5090",                       "team": "billing",   "escalate": False},
    {"email": "I want to cancel order 4471, I'm so frustrated", "team": "billing",   "escalate": True},
    {"email": "My package never showed up and I'm furious",     "team": "logistics", "escalate": True},   # the keyword extractor mis-reads this one
]

def evaluate():
    # per-row: solved = routed to the expected team, escalation matches, never auto-sent (needs_*)
    solved = 0
    for t in SUITE:
        r = process(t["email"])
        if ___:   # TODO: r team == t team AND r escalate == t escalate AND r status starts with "needs_"
            solved += 1
    return solved, len(SUITE)

try:
    for t in SUITE:
        r = process(t['email'])
        print(t['email'][:34], '->', r['team'], '| esc', r['escalate'], '|', r['status'])
    print('score:', evaluate())
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("a delivery email routes to logistics", lambda: process("Where is my order 4471? It's late.")["team"] == "logistics")
expect_true("a refund email routes to billing", lambda: process("Please refund order 5090")["team"] == "billing")
expect_true("a frustrated customer is escalated", lambda: process("I want to cancel order 4471, I'm so frustrated")["escalate"] is True)
expect_true("the delivery draft is grounded (ETA present)", lambda: "Friday" in process("Where is my order 4471? It's late.")["draft"])
expect_true("NO email is ever auto-sent (always needs_*)", lambda: all(process(t["email"])["status"].startswith("needs_") for t in SUITE))
expect_true("evaluate yields a PARTIAL score (the keyword extractor misses one)", lambda: evaluate() == (3, 4))
expect_true("the capstone REUSES the Lab 7.11 gather-only agent", lambda: type(make_email_agent()).__name__ == "CompiledStateGraph")
expect_true("that agent still withholds send_email (cannot auto-send)", lambda: "send_email" not in [t.name for t in gather_tools()])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
Swap the template draft for a REAL model draft (Ollama / Groq) -- the bridge to Module 8. This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
try:
    if ollama_up():
        from langchain_ollama import ChatOllama
        llm = ChatOllama(model="llama3.2:1b")
        email = "Where is my order 4471? It's late."
        reply = llm.invoke("You are a support agent. Draft a short, polite reply (do NOT send): " + email).content
        print("REAL drafted reply:\n", reply)
        print("\nProduction shape: extract -> route -> gather (tools) -> draft (llm) -> validate -> human approves -> send.")
    else:
        print("No Ollama reachable -- skipping the live draft. The offline pipeline above already ran the whole suite")
        print("(extract -> route -> draft -> validate) and returned needs_approval drafts, never auto-sent.")
    print("Next: Module 8 -- when one agent isn't enough, route to specialist AGENTS (the customer-service chatbot).")
except Exception as e:
    print("Live draft skipped:", type(e).__name__)

---
### Key takeaway
You built the email-drafting agent end to end -- extract, route, gather, draft, validate -- and it never sends on its own. That's an agent that does a job you'd pay for. Next: Module 8 orchestrates a team of them.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>